# Huấn Luyện Mô Hình LSTM Dự Báo Cổ Phiếu FPT

Mục tiêu:
- Huấn luyện mạng nơ-ron hồi quy LSTM dự báo tỷ suất sinh lời (log return) của cổ phiếu FPT.
- Sử dụng bộ 27 đặc trưng: chỉ báo kỹ thuật, chỉ báo khối lượng (OBV, CMF, Volume Ratio) và chỉ số thị trường.
- Dùng hàm mất mát Huber để giảm ảnh hưởng của các giá trị ngoại lai.
- Khảo sát kích thước cửa sổ trượt (lookback = 10, 30, 60 ngày).
- Lưu mô hình và bộ chuẩn hóa vào `models/deep_learning/fpt/` phục vụ triển khai REST API.

Notebook chạy được trực tiếp trên máy cục bộ (khoảng 30 giây cho một lần huấn luyện).

## 1. Thiết lập môi trường

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TICKER = 'FPT.VN'
DATA_PATH = BASE_DIR / 'data' / 'interim' / f'{TICKER}_features.csv'
BASELINES_DIR = BASE_DIR / 'models' / 'baselines'
DL_DIR = BASE_DIR / 'models' / 'deep_learning' / 'fpt'
FIG_DIR = BASE_DIR / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DL_DIR.mkdir(parents=True, exist_ok=True)
print('Mã cổ phiếu:', TICKER)
print('Dữ liệu:', DATA_PATH, '| tồn tại:', DATA_PATH.exists())

import shutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'không có, chạy bằng CPU')

## 2. Nạp dữ liệu

In [ ]:
df_fpt = pd.read_csv(DATA_PATH)
print('Kích thước:', df_fpt.shape)
print('Khoảng thời gian:', df_fpt['Date'].iloc[0], 'đến', df_fpt['Date'].iloc[-1])
df_fpt.head()

## 3. Tiền xử lý dữ liệu theo cửa sổ trượt

In [ ]:
def prepare_tensor_data(df, lookback=60, train_ratio=0.70, val_ratio=0.15):
    d = df.copy()
    d['Target_Return'] = d['Log_Return'].shift(-1)
    d.replace([np.inf, -np.inf], np.nan, inplace=True)
    d.dropna(inplace=True)
    d.reset_index(drop=True, inplace=True)

    feature_cols = [c for c in d.columns if c not in ['Date', 'Target_Return']]
    X_raw = d[feature_cols].values
    y_raw = d['Target_Return'].values.reshape(-1, 1)

    n = len(d)
    tr_end, va_end = int(n * train_ratio), int(n * (train_ratio + val_ratio))

    f_scaler, t_scaler = MinMaxScaler(), MinMaxScaler()
    X_tr = f_scaler.fit_transform(X_raw[:tr_end])
    X_va = f_scaler.transform(X_raw[tr_end:va_end])
    X_te = f_scaler.transform(X_raw[va_end:])
    y_tr = t_scaler.fit_transform(y_raw[:tr_end])
    y_va = t_scaler.transform(y_raw[tr_end:va_end])
    y_te = t_scaler.transform(y_raw[va_end:])

    def seq(X, y):
        return (np.array([X[i - lookback:i] for i in range(lookback, len(X))]),
                np.array([y[i] for i in range(lookback, len(X))]))

    return (*seq(X_tr, y_tr), *seq(X_va, y_va), *seq(X_te, y_te), f_scaler, t_scaler, feature_cols)

## 4. Kiến trúc mô hình LSTM

In [ ]:
def build_lstm_model(timesteps, n_features):
    model = Sequential([
        Input(shape=(timesteps, n_features)),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1, activation='linear'),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='huber', metrics=['mae'])
    return model


def evaluate(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    t, p = np.sign(y_true), np.sign(y_pred)
    mask = (t != 0) & (p != 0)
    hit = float((t[mask] == p[mask]).mean() * 100) if mask.any() else 50.0
    return hit, rmse, mae

## 5. Huấn luyện mô hình chính (lookback = 60 ngày)

In [ ]:
LOOKBACK = 60
X_tr, y_tr, X_va, y_va, X_te, y_te, f_scaler, t_scaler, feats = prepare_tensor_data(df_fpt, LOOKBACK)
print('Số mẫu train / val / test:', len(X_tr), len(X_va), len(X_te), '| số đặc trưng:', len(feats))

model_path = DL_DIR / f'{TICKER}_lstm_model.keras'
model = build_lstm_model(LOOKBACK, X_tr.shape[2])
callbacks = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=0),
    ModelCheckpoint(filepath=str(model_path), monitor='val_loss', save_best_only=True, verbose=0),
]
history = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=75, batch_size=32, callbacks=callbacks, verbose=0)

y_pred = t_scaler.inverse_transform(model.predict(X_te, verbose=0)).flatten()
y_true = t_scaler.inverse_transform(y_te).flatten()
hit, rmse, mae = evaluate(y_true, y_pred)
print(f'Số epoch: {len(history.history["loss"])} | Hit Rate = {hit:.2f}% | RMSE = {rmse:.5f} | MAE = {mae:.5f}')

joblib.dump(f_scaler, DL_DIR / f'{TICKER}_feature_scaler.pkl')
joblib.dump(t_scaler, DL_DIR / f'{TICKER}_target_scaler.pkl')

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history.history['loss'], label='Mất mát huấn luyện')
plt.plot(history.history['val_loss'], label='Mất mát kiểm định')
plt.title('Đường học của mô hình LSTM (FPT)')
plt.xlabel('Epoch')
plt.ylabel('Huber loss')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fpt_training_curve.png', dpi=300)
plt.show()

## 6. Khảo sát kích thước cửa sổ trượt (Ablation Study)

In [ ]:
records = []
for lb in [10, 30, 60]:
    a = prepare_tensor_data(df_fpt, lb)
    m = build_lstm_model(lb, a[0].shape[2])
    m.fit(a[0], a[1], validation_data=(a[2], a[3]), epochs=75, batch_size=32, verbose=0,
          callbacks=[EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)])
    p = a[7].inverse_transform(m.predict(a[4], verbose=0)).flatten()
    t = a[7].inverse_transform(a[5]).flatten()
    h, r, ma = evaluate(t, p)
    records.append({'Lookback (ngày)': lb, 'Hit Rate (%)': h, 'RMSE': r, 'MAE': ma})
    print(f'Lookback = {lb:<2} ngày | Hit Rate = {h:.2f}% | RMSE = {r:.5f}')

ablation_df = pd.DataFrame(records)
plt.figure(figsize=(7, 4))
sns.barplot(data=ablation_df, x='Lookback (ngày)', y='Hit Rate (%)', color='#3b82f6')
plt.axhline(50, color='gray', linestyle='--', label='Đoán ngẫu nhiên (50%)')
plt.title('Ảnh hưởng của kích thước cửa sổ trượt (FPT)')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fpt_lookback_ablation.png', dpi=300)
plt.show()
ablation_df.round(5)

## 7. Đóng gói mô hình mặc định cho REST API

In [ ]:
shutil.copy(model_path, DL_DIR / 'best_lstm_model.keras')
shutil.copy(DL_DIR / f'{TICKER}_feature_scaler.pkl', DL_DIR / 'feature_scaler.pkl')
shutil.copy(DL_DIR / f'{TICKER}_target_scaler.pkl', DL_DIR / 'target_scaler.pkl')
print('Đã lưu mô hình mặc định vào', DL_DIR)